# v6 FIX run (GPU 0): continual + gentle per-step noise, from phase 60
Continual run `v4b_continual_s0` continues from phase 60 (Hindi, Chinese, Arabic, ...). After every optimizer step, noise of 0.0007 x each weight matrix's starting std is added (the 'perturb' half of Shrink-and-Perturb; weight decay is the 'shrink'). Question: does the model keep learning after phase 60 instead of breaking down?

**Just Run All.** Output appears below, as in v5. Rerunning resumes from the last saved phase.

In [1]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
os.system("pkill -f 'train_v6.py.*--gpu 0'")  # stop any background job from the earlier notebooks on this GPU
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6_cont_noise0.0007_from60"

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
# get what this run needs (skips what is already here)
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
# 1) the checkpoint we branch from
SRC_RUN, SRC_PHASE = "v4b_continual_s0", 60
if not glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt"):
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
INIT = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")[0]
print("branch from:", INIT)
# 2) a leftover folder from the earlier (crashed) attempt with nothing trained yet -> start clean
done = [p for p in glob.glob(f"runs/{RUN}/ckpt/p*.pt") if int(os.path.basename(p)[1:4]) > SRC_PHASE]
if os.path.isdir(f"runs/{RUN}") and not done:
    import shutil; shutil.rmtree(f"runs/{RUN}"); print("removed empty leftover folder")
# 3) FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
# 4) training data must be here
need = [f"data/en-{l}/train.jsonl" for l in ["hi", "zh", "ar"]] + [f"data/en-{l}/val.jsonl" for l in LANGS]
missing = [p for p in need if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

branch from: runs/v4b_continual_s0/ckpt/p060_ar.pt
removed empty leftover folder
data OK


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "zh", "ar"), init_ckpt=INIT, start_phase=SRC_PHASE,
              n_phases=90, seed=0, time_cap_h=120.0, hf_repo=HF_REPO, noise_per_step=7e-4)
print(C)

RunConfig(run_name='v6_cont_noise0.0007_from60', mode='continual', order=('hi', 'zh', 'ar'), init_ckpt='runs/v4b_continual_s0/ckpt/p060_ar.pt', n_phases=90, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=60, noise_per_step=0.0007, boundary_push=0.0, health_every=1)


In [4]:
D = load_data(train_langs=["hi", "zh", "ar"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  ar: 998,000 train / 2,000 val
data loaded in 55s


In [ ]:
print(1)

In [ ]:
model = train_run(C, D)

 RUN v6_cont_noise0.0007_from60 | mode continual | 90 phases after phase 0 | device cuda
 phase order: init (loaded) → hi → zh → ar → hi → zh → ar → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
BRANCH: starting from runs/v4b_continual_s0/ckpt/p060_ar.pt (phase 60, ar); next phase 61

 v6: noise_per_step = 0.0007 (x each matrix's starting std) | boundary_push = 0.0 (x each matrix's current std, at the start of every phase)

▶ PHASE 61/90 | training hi (993,016 pairs)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[W1026 22:36:03.670467703 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1920991232 bytes (free: 1354498048, total: 25250627584).


  (HF backup failed, training continues: Network error: Request middleware error: error sending request for url (https://huggingface.co/api/models/s8n29/cml-lang-self-checkpoints/xet-write-token/main))
  [p61 hi] step    500 | lr 2.0e-04 | train 3.684 | val 1.913 (best 1.913) | window Δ n/a | stable | 74k tok/s | phase 1m39s | run ETA: after first phase | health {'wnorm': 228.78, 'rank_enc1': 1061.7, 'rank_dec1': 613.3, 'rarely_active': 0.1568}
  [p61 hi] step  1,000 | lr 2.0e-04 | train 2.832 | val 1.623 (best 1.623) | window Δ n/a | stable | 76k tok/s | phase 3m15s | run ETA: after first phase | health {'wnorm': 228.937, 'rank_enc1': 1086.6, 'rank_dec1': 691.8, 'rarely_active': 0.1496}
  [p61 hi] step  1,500 | lr 2.0e-04 | train 2.654 | val 1.511 (best 1.511) | window Δ n/a | stable | 77k tok/s | phase 4m50s | run ETA: after first phase | health {'wnorm': 229.105, 'rank_enc1': 1081.3, 'rank_dec1': 707.4, 'rarely_active': 0.1497}
  [p61 hi] step  2,000 | lr 2.0e-04 | train 2.530 | val

...

  (HF backup failed, training continues: Previous task error: Network error: Request middleware error: error sending request for url (https://huggingface.co/api/models/s8n29/cml-lang-self-checkpoints/xet-write-token/main))
  [p85 hi] step    500 | lr 2.0e-04 | train 5.187 | val 3.540 (best 3.540) | window Δ n/a | stable | 75k tok/s | phase 1m37s | run ETA ~3h53m | health {'wnorm': 236.739, 'rank_enc1': 630.5, 'rank_dec1': 435.1, 'rarely_active': 0.4529}
  [p85 hi] step  1,000 | lr 2.0e-04 | train 4.389 | val 3.225 (best 3.225) | window Δ n/a | stable | 77k tok/s | phase 3m12s | run ETA ~3h53m | health {'wnorm': 236.529, 'rank_enc1': 666.5, 'rank_dec1': 441.4, 'rarely_active': 0.4505}
  [p85 hi] step  1,500 | lr 2.0e-04 | train 4.174 | val 3.039 (best 3.039) | window Δ n/a | stable | 77k tok/s | phase 4m47s | run ETA ~3h53m | health {'wnorm': 236.442, 'rank_enc1': 665.4, 'rank_dec1': 456.6, 'rarely_active': 0.4472}
  [p85 hi] step  2,000 | lr 2.0e-04 | train 4.050 | val 2.948 (best 2.94

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  (HF backup failed, training continues: Previous task error: Network error: Request middleware error: error sending request for url (https://huggingface.co/api/models/s8n29/cml-lang-self-checkpoints/xet-write-token/main))
  [p87 ar] step    500 | lr 2.0e-04 | train 5.506 | val 3.918 (best 3.918) | window Δ n/a | stable | 71k tok/s | phase 1m36s | run ETA ~2h41m | health {'wnorm': 236.708, 'rank_enc1': 367.1, 'rank_dec1': 315.3, 'rarely_active': 0.5042}
  [p87 ar] step  1,000 | lr 2.0e-04 | train 4.628 | val 3.576 (best 3.576) | window Δ n/a | stable | 72k tok/s | phase 3m11s | run ETA ~2h41m | health {'wnorm': 236.579, 'rank_enc1': 395.3, 'rank_dec1': 338.8, 'rarely_active': 0.5034}
  [p87 ar] step  1,500 | lr 2.0e-04 | train 4.460 | val 3.445 (best 3.445) | window Δ n/a | stable | 71k tok/s | phase 4m49s | run ETA ~2h41m | health {'wnorm': 236.548, 'rank_enc1': 327.0, 'rank_dec1': 328.6, 'rarely_active': 0.496}
  [p87 ar] step  2,000 | lr 2.0e-04 | train 4.351 | val 3.339 (best 3.339

KeyboardInterrupt: 

In [ ]:
print_run_summary(RUN)

 SUMMARY v6_cont_noise0.0007_from60: 27 phases (incl. phase 0)
--------------------------------------------------------------------------------------
  ph lang   steps      stop | chrF hi chrF zh chrF ar | val hi val zh val ar   (script-restricted)
  60   ar  10,500   plateau |     7.9     2.8    52.3 |  5.644  5.334  1.842
  61   hi  14,000   plateau |    51.4     1.9    11.5 |  1.268  6.243  5.720
  62   zh   9,000   plateau |    25.2    26.1    17.1 |  4.274  2.025  5.735
  63   ar  10,500   plateau |     6.4     3.2    51.7 |  5.656  5.214  1.853
  64   hi  13,000   plateau |    50.8     2.1    11.5 |  1.290  6.066  5.521
  65   zh   8,500   plateau |    25.5    25.2    18.8 |  4.097  2.038  5.538
  66   ar  10,500   plateau |     6.1     2.6    51.9 |  5.584  5.202  1.884
  67   hi  13,500   plateau |    50.8     1.7    10.6 |  1.316  6.095  5.516
  68   zh  10,000   plateau |    24.4    25.2    18.0 |  4.303  2.051  5.706
  69   ar  12,000   plateau |     3.8     1.8    51.4 |  5